# V1: super-stupid broch wave test

This notebook is intentionally simple. Its job is to make the statistical idea visible before we let archaeology, Bayesian models, maritime routes, or clever optimization complicate it.

For each site, imagine a vector of candidate dates:

$$D_i = \{t_{i1}, t_{i2}, \ldots, t_{in_i}\}$$

A **slice** chooses exactly one date from each site. We score that slice with **Kendall's tau** between distance from a fixed hypothetical origin and date.

- $\tau \approx +1$: farther sites are very consistently later.
- $\tau \approx 0$: no monotonic spatial ordering.
- $\tau \approx -1$: farther sites are consistently earlier.

This V1 does **not** claim that any chosen date is a construction date. It is an educational stress test of the data shape.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)


## 1. Build synthetic worlds where we know the truth

Before touching Scotland, make the algorithm face three toy worlds:

1. **obvious_wave** — date increases strongly with distance;
2. **weak_wave** — the same basic relationship buried in much more noise;
3. **garbage** — dates are generated independently of distance.

Each synthetic site gets several candidate dates around its hidden `true_date`. Negative years mean BC only for convenience in this notebook.

In [ ]:
def make_world(kind, n_sites=12, dates_per_site=5, seed=42):
    local_rng = np.random.default_rng(seed)
    distances = np.linspace(0, 550, n_sites)

    if kind == "obvious_wave":
        true_dates = -400 + 0.55 * distances + local_rng.normal(0, 12, n_sites)
        observation_noise = 25
    elif kind == "weak_wave":
        true_dates = -400 + 0.30 * distances + local_rng.normal(0, 70, n_sites)
        observation_noise = 60
    elif kind == "garbage":
        true_dates = local_rng.uniform(-425, 50, n_sites)
        observation_noise = 60
    else:
        raise ValueError(f"Unknown world: {kind}")

    sites = []
    for i, (distance, true_date) in enumerate(zip(distances, true_dates)):
        dates = true_date + local_rng.normal(0, observation_noise, dates_per_site)
        sites.append({
            "name": f"Site {i + 1:02d}",
            "distance_km": float(distance),
            "true_date": float(true_date),
            "dates": np.asarray(dates, dtype=float),
        })

    return sites


worlds = {
    "obvious_wave": make_world("obvious_wave", seed=1),
    "weak_wave": make_world("weak_wave", seed=2),
    "garbage": make_world("garbage", seed=3),
}


## 2. The deliberately dumb culling algorithm

Pick one candidate date uniformly at random from each site's vector. Then compute Kendall's tau.

Uniform choice is knowingly archaeologically wrong. A later version can weight samples by context. Right now we want to understand the mechanics.

In [ ]:
def choose_slice(sites, rng):
    return np.array([rng.choice(site["dates"]) for site in sites], dtype=float)


def wave_score(sites, chosen_dates):
    distances = np.array([site["distance_km"] for site in sites], dtype=float)
    return float(kendalltau(distances, chosen_dates).statistic)


def sample_wave_scores(sites, n=10_000, seed=42):
    local_rng = np.random.default_rng(seed)
    scores = np.empty(n, dtype=float)

    for i in range(n):
        chosen_dates = choose_slice(sites, local_rng)
        scores[i] = wave_score(sites, chosen_dates)

    return scores


## 3. Does the dumb test distinguish worlds we already understand?

If this fails here, it has no business touching real archaeology.

In [ ]:
scores_by_world = {
    name: sample_wave_scores(sites, n=20_000, seed=RNG_SEED)
    for name, sites in worlds.items()
}

summary = pd.DataFrame({
    name: {
        "mean_tau": np.mean(scores),
        "median_tau": np.median(scores),
        "p_tau_gt_0": np.mean(scores > 0),
        "p_tau_gt_0_4": np.mean(scores > 0.4),
        "min_tau": np.min(scores),
        "max_tau": np.max(scores),
    }
    for name, scores in scores_by_world.items()
}).T

summary


In [ ]:
for name, scores in scores_by_world.items():
    plt.figure()
    plt.hist(scores, bins=40)
    plt.axvline(0, linewidth=1)
    plt.title(name)
    plt.xlabel("Kendall tau")
    plt.ylabel("Random slices")
    plt.show()


## 4. Null experiment: break geography while preserving each vector

Now shuffle **whole date vectors** among the existing site distances. This preserves the vector contents and vector sizes but deliberately destroys the original geography/date association.

Then apply the exact same random-slice procedure.

This is still a crude null model, but it gives us the right conceptual comparison: **how easily can spatially randomized data look wave-like under the same rules?**

In [ ]:
def permute_date_vectors(sites, rng):
    permutation = rng.permutation(len(sites))
    vectors = [sites[j]["dates"] for j in permutation]

    return [
        {
            **site,
            "dates": vectors[i],
        }
        for i, site in enumerate(sites)
    ]


def sample_null_scores(sites, n=20_000, seed=43):
    local_rng = np.random.default_rng(seed)
    scores = np.empty(n, dtype=float)

    for i in range(n):
        shuffled = permute_date_vectors(sites, local_rng)
        chosen_dates = choose_slice(shuffled, local_rng)
        scores[i] = wave_score(shuffled, chosen_dates)

    return scores


In [ ]:
real_scores = scores_by_world["obvious_wave"]
null_scores = sample_null_scores(worlds["obvious_wave"], n=20_000)

pd.DataFrame({
    "real": pd.Series(real_scores).describe(),
    "null": pd.Series(null_scores).describe(),
})


In [ ]:
plt.figure()
plt.hist(null_scores, bins=40, alpha=0.6, label="null / shuffled")
plt.hist(real_scores, bins=40, alpha=0.6, label="real arrangement")
plt.axvline(0, linewidth=1)
plt.xlabel("Kendall tau")
plt.ylabel("Slices")
plt.legend()
plt.show()


## 5. Real-data adapter — intentionally unfinished

Do **not** paste the 21-site corpus into this cell until we have a reproducible table of site vectors and an explicitly chosen origin/distance definition.

When Codex wires real data in, the minimal required shape is:

```python
real_sites = [
    {
        "name": "Example",
        "distance_km": 123.4,
        "dates": np.array([-310, -250, -180]),
    },
]
```

Then the exact same functions above should run unchanged:

```python
real_scores = sample_wave_scores(real_sites)
null_scores = sample_null_scores(real_sites)
```

If we have to rewrite the scoring logic specially for the archaeological data, stop and document why.

## 6. What this notebook can and cannot tell us

**It can ask:**

- do random slices of the observed vectors tend to preserve a monotonic spatial ordering?
- does that behavior differ from the same vectors after geography is permuted?
- can our simple statistic distinguish synthetic signal from synthetic garbage?

**It cannot yet tell us:**

- which dates actually constrain broch construction;
- whether the chosen origin is historically meaningful;
- whether Euclidean distance approximates Iron Age interaction;
- whether archaeological sampling bias created an apparent spatial pattern;
- whether any detected ordering represents cultural diffusion.

V1 is a test harness for our reasoning, not a result.